# Fundlenz v2 — multi-index retrieval demo

This notebook walks through the v2 pipeline (inverted exact-match + text ANN + deterministic rerank) with the example queries from the plan. It loads `state.composite` from the persisted index — run `python -m scripts.migrate_v2` first to populate it from `data/uploads/`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from app import state
from app.retrieval.router import classify_intent
from app.retrieval.orchestrator import retrieve_v2

state.composite.load_or_init()
state.composite.stats()

## 1. "What's in fund.csv?" — synopsis chunk should win

In [ ]:
q = "What's in Fund-Performance-01-May-2026--1104.xlsx?"
plan = classify_intent(q)
print('intent:', plan.intent, '| canonical_id:', plan.canonical_id)
for c in retrieve_v2(q, state.composite, k=5):
    print(f"  {c['chunk_type']:20s} score={c['_score']:.3f} {c.get('text', '')[:80]}")

## 2. "List all schemes" — enumeration chunk should win

In [ ]:
q = "List all schemes"
plan = classify_intent(q)
print('intent:', plan.intent)
for c in retrieve_v2(q, state.composite, k=3):
    print(f"  {c['chunk_type']:20s} score={c['_score']:.3f} {c.get('text', '')[:120]}")

## 3. "Summarize fund.csv" — tabular_summary expected

In [ ]:
q = "Summarize Fund-Performance-01-May-2026--1104.xlsx"
plan = classify_intent(q)
print('intent:', plan.intent)
for c in retrieve_v2(q, state.composite, k=3):
    print(f"  {c['chunk_type']:20s} score={c['_score']:.3f}")

## 4. "What is the NAV regular of <Fund X>?" — point lookup, exact-id match wins

In [ ]:
q = "What is the NAV regular of HDFC Top 100 Fund?"
plan = classify_intent(q)
print('intent:', plan.intent, '| entities:', plan.raw_entity_phrases)
for c in retrieve_v2(q, state.composite, k=3):
    print(f"  {c['chunk_type']:18s} canon={c.get('canonical_id')!s:30s} score={c['_score']:.3f}")

## 5. "List funds with NAV between 100 and 200 on 2025-04-30" — numeric_threshold + post-compute

In [ ]:
q = "List funds with NAV greater than 100"
plan = classify_intent(q)
print('intent:', plan.intent, '| threshold:', plan.threshold)
for c in retrieve_v2(q, state.composite, k=5):
    print(f"  {c['chunk_type']:18s} canon={c.get('canonical_id')!s:30s} score={c['_score']:.3f}")

print('\nFor numeric thresholds the deterministic pandas path via query_table is the source of truth:')
from app.analysis.query import query_table
fid = next(iter(state.dataframes_by_file_id))
df = state.dataframes_by_file_id[fid]
result = query_table(df, filters=[{'column': 'NAV Regular', 'op': '>', 'value': 100}], limit=5)
result.head()